# Sarcasm Detection — Exploratory Data Analysis

**Dataset:** News Headlines Dataset for Sarcasm Detection (Misra & Arora, 2019)  
**Goal:** Understand the raw data before modeling — class balance, text length, vocabulary, top n-grams.

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## 1. Load Raw Data

In [ ]:
records = []
with open('../data/raw/sarcasm_headlines.json', 'r') as f:
    for line in f:
        records.append(json.loads(line))

df = pd.DataFrame(records)[['headline', 'is_sarcastic']]
df.columns = ['text', 'label']
print(f'Total samples: {len(df):,}')
df.head()

## 2. Class Balance

In [ ]:
counts = df['label'].value_counts()
labels = ['Non-sarcastic', 'Sarcastic']
print(counts)

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(labels, [counts[0], counts[1]], color=['#4C72B0', '#DD8452'])
ax.set_title('Class Distribution')
ax.set_ylabel('Count')
for i, v in enumerate([counts[0], counts[1]]):
    ax.text(i, v + 100, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center')
plt.tight_layout()
plt.show()

**Finding:** The dataset is nearly balanced (52.4% non-sarcastic / 47.6% sarcastic), so accuracy is a meaningful metric and no oversampling is needed.

## 3. Text Length Distribution

In [ ]:
df['word_count'] = df['text'].str.split().str.len()
df['char_count'] = df['text'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for label, name, color in [(0, 'Non-sarcastic', '#4C72B0'), (1, 'Sarcastic', '#DD8452')]:
    subset = df[df['label'] == label]
    axes[0].hist(subset['word_count'], bins=30, alpha=0.6, label=name, color=color)
    axes[1].hist(subset['char_count'], bins=30, alpha=0.6, label=name, color=color)

axes[0].set_title('Word Count Distribution')
axes[0].set_xlabel('Words per headline')
axes[0].legend()

axes[1].set_title('Character Count Distribution')
axes[1].set_xlabel('Characters per headline')
axes[1].legend()

plt.tight_layout()
plt.show()

print(df.groupby('label')[['word_count', 'char_count']].mean())

**Finding:** Sarcastic headlines tend to be slightly longer on average, consistent with the rhetorical elaboration typical of irony.

## 4. Duplicate Check

In [ ]:
dups = df.duplicated(subset=['text']).sum()
print(f'Duplicate headlines: {dups}')
# → 142 duplicates found; removed in make_dataset.py

## 5. Top Unigrams — Sarcastic vs Non-sarcastic

In [ ]:
def top_ngrams(texts, n=1, k=20):
    vec = CountVectorizer(ngram_range=(n, n), stop_words='english', max_features=5000)
    X = vec.fit_transform(texts)
    counts = X.sum(axis=0).A1
    terms = vec.get_feature_names_out()
    return sorted(zip(terms, counts), key=lambda x: -x[1])[:k]

sarcastic_texts = df[df['label'] == 1]['text']
nonsarcastic_texts = df[df['label'] == 0]['text']

top_s = top_ngrams(sarcastic_texts, n=1)
top_ns = top_ngrams(nonsarcastic_texts, n=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, title, color in [
    (axes[0], top_s,  'Top 20 Unigrams — Sarcastic',     '#DD8452'),
    (axes[1], top_ns, 'Top 20 Unigrams — Non-sarcastic', '#4C72B0')
]:
    terms, cnts = zip(*data)
    ax.barh(terms[::-1], cnts[::-1], color=color)
    ax.set_title(title)
    ax.set_xlabel('Count')

plt.tight_layout()
plt.show()

## 6. Top Bigrams

In [ ]:
top_s2  = top_ngrams(sarcastic_texts, n=2, k=15)
top_ns2 = top_ngrams(nonsarcastic_texts, n=2, k=15)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, data, title, color in [
    (axes[0], top_s2,  'Top 15 Bigrams — Sarcastic',     '#DD8452'),
    (axes[1], top_ns2, 'Top 15 Bigrams — Non-sarcastic', '#4C72B0')
]:
    terms, cnts = zip(*data)
    ax.barh(terms[::-1], cnts[::-1], color=color)
    ax.set_title(title)
    ax.set_xlabel('Count')

plt.tight_layout()
plt.show()

## 7. Source Distribution

Sarcastic headlines come from *The Onion*; non-sarcastic from *HuffPost*. The lexical differences observed above reflect the stylistic gap between satirical and news writing — which is exactly what all three models exploit.

In [ ]:
print('Summary statistics')
print('=' * 40)
print(f"Total samples:       {len(df):>8,}")
print(f"Sarcastic:           {counts[1]:>8,} ({counts[1]/len(df)*100:.1f}%)")
print(f"Non-sarcastic:       {counts[0]:>8,} ({counts[0]/len(df)*100:.1f}%)")
print(f"Duplicates removed:  {dups:>8,}")
print(f"Avg words (sarc):    {df[df.label==1].word_count.mean():>8.1f}")
print(f"Avg words (non):     {df[df.label==0].word_count.mean():>8.1f}")